# 01 — Phase 2 Core: Target, QA, Cleaning, Baseline Transfer

This is a cleaned version of the previous `Mod2_phase2_v2_...` notebook. It does not delete or modify the original file.

**Sequence:** This notebook is run first. Then `02_External_Features_Final.ipynb`, then `03_Transfer_Experiments_Final.ipynb` are run.

**Fixed target:** `Mod2` + `Nig` + `Intensity_MEA` + period `22` (JJA), 2001–2020.

**Scientific security:** Duplicate target keys are not automatically resolved using “keep first line”; all ambiguous keys are quarantined. JJA monthly–seasonal difference is not automatically classified as true/false; it is reported and awaits human approval. Notebook cell outputs have been intentionally cleaned; New outputs are only generated when you run it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, math, hashlib, shutil, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
DRIVE_ROOT = Path('/content/drive/MyDrive')
# Prefer the clean GitHub-style repository folder; retain compatibility with the previous tao_documents layout.
_REPO_CANDIDATES = [
    DRIVE_ROOT / 'Urban_Heat_Transfer_COMPLETE_BUNDLE' / 'urban-heat-transfer',
    DRIVE_ROOT / 'urban-heat-transfer',
    DRIVE_ROOT / 'BIRLESIK_PROJE_PAKETI' / 'urban-heat-transfer',
]
PROJECT_DIR = next((p for p in _REPO_CANDIDATES if (p / 'notebooks').is_dir() and (p / 'data').is_dir()), None)
LEGACY_LAYOUT = PROJECT_DIR is None
if PROJECT_DIR is None:
    PROJECT_DIR = DRIVE_ROOT / 'tao_documents' if (DRIVE_ROOT / 'tao_documents').is_dir() else DRIVE_ROOT
if LEGACY_LAYOUT:
    PHASE2_ROOT = PROJECT_DIR / 'phase2_outputs'
    BACKUP_ROOT = PROJECT_DIR / 'previous_output_backups'
else:
    PHASE2_ROOT = PROJECT_DIR / 'outputs' / 'phase2_outputs'
    BACKUP_ROOT = PROJECT_DIR / 'outputs' / 'previous_output_backups'
DOCS_DIR = PROJECT_DIR / 'docs'
AUDIT_DIR = PHASE2_ROOT / 'audit_reports'
for p in (PHASE2_ROOT, DOCS_DIR, AUDIT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Preserve generated artefacts before a repeat run. Original source datasets are never modified.
PRE_RUN_BACKUP_DIR = BACKUP_ROOT / f'core_{RUN_ID}'
existing_outputs = []
for folder in [PHASE2_ROOT, DOCS_DIR]:
    if folder.exists():
        existing_outputs.extend(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv','.md','.json'})
if existing_outputs:
    for old_path in existing_outputs:
        try:
            rel = old_path.relative_to(PROJECT_DIR)
            dest = PRE_RUN_BACKUP_DIR / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(old_path, dest)
        except OSError as exc:
            raise RuntimeError(f'Could not preserve previous output {old_path}: {exc}')
    print(f'Archived {len(existing_outputs):,} previous output files to {PRE_RUN_BACKUP_DIR}')

# Locate the input data. In this repository, the expected path is data/raw/UHII_dataset/.
# The additional candidates preserve compatibility with the older package layout.
DATA_ROOT_CANDIDATES = [
    PROJECT_DIR / 'data' / 'raw',
    PROJECT_DIR / 'data',
    PROJECT_DIR,
    DRIVE_ROOT / '24821538',
    PROJECT_DIR / '24821538',
    DRIVE_ROOT / 'data' / '24821538',
    PROJECT_DIR / 'data' / '24821538',
    DRIVE_ROOT,
]
UHII_FOLDER = next((candidate / 'UHII_dataset' for candidate in DATA_ROOT_CANDIDATES
                    if (candidate / 'UHII_dataset').is_dir()), None)
if UHII_FOLDER is None:
    expected = [str(candidate / 'UHII_dataset') for candidate in DATA_ROOT_CANDIDATES]
    raise FileNotFoundError('Could not locate the extracted UHII_dataset folder. Checked:\n' + '\n'.join(expected))
DATA_ROOT = UHII_FOLDER.parent

YEAR_MIN, YEAR_MAX = 2001, 2020
YEARS = list(range(YEAR_MIN, YEAR_MAX + 1))
TARGET_PROD, TARGET_TIME, TARGET_COL, TARGET_PERIOD = 'Mod2', 'Nig', 'Intensity_MEA', 22
TARGET_UNIT = 'degC'
TEST_FRACTION = 0.20
TEST_LOCK_SEED = 20261017
ADAPTATION_SEEDS = list(range(10))
FEW_SHOT_RATIOS = [0.0, 0.01, 0.05, 0.10, 0.20, 0.50]

# Protocol decision: the published period-22 seasonal file is the primary target.
# The unweighted mean of monthly SUHII is a diagnostic, not a required identity.
# Large residuals are flagged and retained unless separate evidence establishes a source/key error.
JJA_TARGET_RULE_APPROVED_BY_PROTOCOL = True

assert DATA_ROOT.exists(), f'Dataset root not found: {DATA_ROOT}'
assert UHII_FOLDER.exists(), f'UHII folder not found: {UHII_FOLDER}'
print(f'Run ID: {RUN_ID}')
print(f'Project directory: {PROJECT_DIR}')
print(f'Dataset folder: {UHII_FOLDER}')
print(f'Outputs: {PHASE2_ROOT}')
print(f'Target: {TARGET_PROD}/{TARGET_TIME}/{TARGET_COL}/period={TARGET_PERIOD}, years={YEAR_MIN}-{YEAR_MAX}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archived 14 previous output files to /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/outputs/previous_output_backups/core_20261018_002711
Run ID: 20261018_002711
Project directory: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer
Dataset folder: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/data/raw/UHII_dataset
Outputs: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/outputs/phase2_outputs
Target: Mod2/Nig/Intensity_MEA/period=22, years=2001-2020


## Step 1 — Input paths and city manifest

Several known paths are attempted for mapping and the disputed city list. If none are found, the cell stops, stating what is missing. The 13 ID list is retrieved from the file; if the file cannot be found, the list recorded in the previous notebook is used as a fallback, and this is noted in the report.

In [ ]:
def first_existing(paths, label, required=True):
    for item in paths:
        p = Path(item)
        if p.exists():
            print(f'{label}: {p}')
            return p
    if required:
        raise FileNotFoundError(f'{label} bulunamadı. Denenen yollar:\n' + '\n'.join(map(str, paths)))
    print(f'{label}: not found; optional file will not be used.')
    return None

MAPPING_PATH = first_existing([
    PROJECT_DIR / 'data/external_features/China_Turkiye_city_mapping.csv',
    PROJECT_DIR / 'data/reference/China_Turkiye_city_mapping.csv',
    PROJECT_DIR / 'external_features/China_Turkiye_city_mapping.csv',
    PROJECT_DIR / 'China_Turkiye_city_mapping.csv',
    PROJECT_DIR / 'data/China_Turkiye_city_mapping.csv',
    DRIVE_ROOT / 'external_features/China_Turkiye_city_mapping.csv',
    DRIVE_ROOT / 'China_Turkiye_city_mapping.csv',
    DRIVE_ROOT / 'china-turkiye-urban-heat-transfer-main/china-turkiye-urban-heat-transfer-main/data/China_Turkiye_city_mapping.csv',
    DRIVE_ROOT / 'china-turkiye-urban-heat-transfer-main/data/China_Turkiye_city_mapping.csv',
], 'City mapping')
DISPUTED_PATH = first_existing([
    PROJECT_DIR / 'data/audit_reports/Country_Mapping_13_Row_Comparison.csv',
    PROJECT_DIR / 'data/audit_reports/historical_user_audits/Country_Mapping_13_Row_Comparison.csv',
    PROJECT_DIR / 'audit_reports/Country_Mapping_13_Row_Comparison.csv',
    PROJECT_DIR / 'phase2_outputs/audit_reports/Country_Mapping_13_Row_Comparison.csv',
    PROJECT_DIR / 'Country_Mapping_13_Row_Comparison.csv',
    DRIVE_ROOT / 'audit_reports/Country_Mapping_13_Row_Comparison.csv',
    DRIVE_ROOT / 'china-turkiye-urban-heat-transfer-main/china-turkiye-urban-heat-transfer-main/data/audit_reports/Country_Mapping_13_Row_Comparison.csv',
    DRIVE_ROOT / 'china-turkiye-urban-heat-transfer-main/data/audit_reports/Country_Mapping_13_Row_Comparison.csv',
], 'Disputed-ID review', required=False)

# This fallback list is taken from the previous notebook's printed audit output.
# Prefer the source CSV whenever it is available.
KNOWN_DISPUTED_IDS = [3, 62, 64, 601, 617, 2133, 4586, 4785, 5253, 7288, 8344, 8486, 9175]
if DISPUTED_PATH is not None:
    disputed_df = pd.read_csv(DISPUTED_PATH)
    if 'UrbanId' not in disputed_df.columns:
        raise ValueError(f'{DISPUTED_PATH} has no UrbanId column; columns={disputed_df.columns.tolist()}')
    disputed_ids = sorted(pd.to_numeric(disputed_df['UrbanId'], errors='raise').astype(int).unique().tolist())
    disputed_source = str(DISPUTED_PATH)
else:
    disputed_ids = KNOWN_DISPUTED_IDS.copy()
    disputed_source = 'fallback IDs copied from previously inspected notebook output; verify before final sign-off'
if len(disputed_ids) != 13:
    raise ValueError(f'Expected 13 disputed UrbanIds, found {len(disputed_ids)}: {disputed_ids}')

mapping = pd.read_csv(MAPPING_PATH)
required_mapping_cols = {'UrbanId', 'Country', 'Longitude', 'Latitude', 'Area'}
missing_cols = required_mapping_cols - set(mapping.columns)
if missing_cols:
    raise ValueError(f'Mapping is missing required columns {sorted(missing_cols)}; available={mapping.columns.tolist()}')
mapping['UrbanId'] = pd.to_numeric(mapping['UrbanId'], errors='raise').astype(int)
if mapping['UrbanId'].duplicated().any():
    dup = mapping.loc[mapping['UrbanId'].duplicated(keep=False), 'UrbanId'].tolist()[:20]
    raise ValueError(f'Mapping has duplicate UrbanId rows; resolve before continuing. Examples: {dup}')

country_original = mapping['Country'].astype('string').str.strip()
country_normalized = country_original.str.casefold().map({
    'turkiye':'Türkiye', 'türkiye':'Türkiye', 'turkey':'Türkiye',
    'china':'China', 'prc':'China', "people's republic of china":'China'
})
mapping['Country'] = country_normalized.where(country_normalized.notna(), country_original)
manifest = mapping.copy()
manifest['eligibility_status'] = 'INCLUDED'
manifest['disputed_id_flag'] = manifest['UrbanId'].isin(disputed_ids)
manifest['exclusion_reason'] = ''
manifest['mapping_evidence'] = f'Mapping file: {MAPPING_PATH}'
manifest.loc[manifest['disputed_id_flag'], 'eligibility_status'] = 'QUARANTINED'
manifest.loc[manifest['disputed_id_flag'], 'exclusion_reason'] = 'Disputed country/coordinate assignment; needs human review'
unknown = ~manifest['Country'].isin(['China', 'Türkiye'])
manifest.loc[unknown, 'eligibility_status'] = 'QUARANTINED'
manifest.loc[unknown, 'exclusion_reason'] = 'Country label is not China or Türkiye'

manifest_path = PHASE2_ROOT / 'analysis_city_manifest.csv'
manifest.to_csv(manifest_path, index=False)
review = manifest[manifest['disputed_id_flag']].copy()
review['decision'] = 'QUARANTINE'
review['decision_reason'] = 'Do not assign a country from nearest-country matching alone; resolve with authoritative evidence/geometries.'
review['disputed_source'] = disputed_source
review.to_csv(PHASE2_ROOT / 'disputed_urbanid_review.csv', index=False)

eligible_manifest = manifest[manifest['eligibility_status'].eq('INCLUDED')].copy()
counts = eligible_manifest['Country'].value_counts().to_dict()
print(f'Mapping rows: {len(manifest):,}; disputed IDs quarantined: {int(manifest.disputed_id_flag.sum())}')
print(f'Eligible city counts: {counts}')
print(f'Expected only if mapping version matches the reviewed inventory: China=1441, Türkiye=180, total=1621')
print(f'Saved: {manifest_path}')

City mapping: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/data/external_features/China_Turkiye_city_mapping.csv
Disputed-ID review: not found; optional file will not be used.
Mapping rows: 1,634; disputed IDs quarantined: 13
Eligible city counts: {'China': 1441, 'Türkiye': 180}
Expected only if mapping version matches the reviewed inventory: China=1441, Türkiye=180, total=1621
Saved: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/outputs/phase2_outputs/analysis_city_manifest.csv


## Step 2 — Definitive source filter and raw target

`Mod2_Nig_YYYY_PERIOD.csv` files are parsed using regular expressions. Only 2001–2020 and periods 6, 7, 8, and 22 are loaded. The primary target table contains only period 22. Missing targets are not silently filled.

In [ ]:
name_re = re.compile(r'^Mod2_Nig_(\d{4})_(\d{1,2})\.csv$')
inventory_rows = []
for p in sorted(UHII_FOLDER.iterdir()):
    if not p.is_file() or not p.name.lower().endswith('.csv'):
        continue
    m = name_re.match(p.name)
    if not m:
        if 'Mod2_Nig' in p.name or 'AMod2_Nig' in p.name or 'SMod2_Nig' in p.name:
            inventory_rows.append({'filename': p.name, 'bytes': p.stat().st_size, 'selected': False,
                                   'year': None, 'period': None, 'status': 'EXCLUDED_FILENAME',
                                   'note': 'excluded by exact Mod2_Nig_YYYY_PERIOD.csv filename match'})
        continue
    yr, period = int(m.group(1)), int(m.group(2))
    selected = yr in YEARS and period in [6, 7, 8, TARGET_PERIOD]
    inventory_rows.append({'filename': p.name, 'bytes': p.stat().st_size, 'selected': selected,
                           'year': yr, 'period': period,
                           'status': 'SELECTED' if selected else 'OUTSIDE_CONFIGURED_YEAR_OR_PERIOD',
                           'note': 'selected' if selected else 'outside configured year/period'})

actual_files = set()
for row in inventory_rows:
    if row.get('selected'):
        actual_files.add((int(row['year']), int(row['period'])))
expected_seasonal_pairs = {(y, TARGET_PERIOD) for y in YEARS}
expected_monthly_pairs = {(y, m) for y in YEARS for m in [6, 7, 8]}
missing_seasonal_pairs = sorted(expected_seasonal_pairs - actual_files)
missing_monthly_pairs = sorted(expected_monthly_pairs - actual_files)

existing_names = {str(r['filename']) for r in inventory_rows}
for year, period in missing_seasonal_pairs + missing_monthly_pairs:
    expected_name = f'Mod2_Nig_{year}_{period}.csv'
    if expected_name not in existing_names:
        inventory_rows.append({'filename': expected_name, 'bytes': 0, 'selected': False,
                               'year': year, 'period': period, 'status': 'MISSING_EXPECTED_FILE',
                               'note': 'expected input file not found; recorded instead of silently ignored'})

inventory = pd.DataFrame(inventory_rows)
inventory.to_csv(PHASE2_ROOT / 'phase2_file_inventory.csv', index=False)
selected_files = [r['filename'] for r in inventory_rows if r.get('selected')]
if not any(int(r['period']) == TARGET_PERIOD for r in inventory_rows if r.get('selected')):
    raise FileNotFoundError('No exact Mod2_Nig_YYYY_22.csv seasonal target files were found.')
if missing_seasonal_pairs:
    print(f'BLOCKER: {len(missing_seasonal_pairs)} of {len(YEARS)} expected JJA target files are missing: {missing_seasonal_pairs[:20]}')
if missing_monthly_pairs:
    print(f'WARNING: {len(missing_monthly_pairs)} monthly files are missing; the precomputed JJA target can still be audited, but composite validation is incomplete.')

rows = []
for fname in sorted(selected_files):
    m = name_re.match(fname)
    yr, period = int(m.group(1)), int(m.group(2))
    tmp = pd.read_csv(UHII_FOLDER / fname)
    needed = {'UrbanId', TARGET_COL}
    if not needed.issubset(tmp.columns):
        raise ValueError(f'{fname} missing columns {needed - set(tmp.columns)}')
    tmp['UrbanId'] = pd.to_numeric(tmp['UrbanId'], errors='coerce')
    tmp[TARGET_COL] = pd.to_numeric(tmp[TARGET_COL], errors='coerce')
    if tmp['UrbanId'].isna().any():
        raise ValueError(f'{fname} has invalid UrbanId values')
    if not np.allclose(tmp['UrbanId'].to_numpy(), np.round(tmp['UrbanId'].to_numpy())):
        raise ValueError(f'{fname} contains non-integer UrbanId values')
    tmp['UrbanId'] = tmp['UrbanId'].astype(int)
    # Infinity is not a usable target. Record as missing instead of allowing it into models.
    tmp[TARGET_COL] = tmp[TARGET_COL].replace([np.inf, -np.inf], np.nan)
    tmp['year'] = yr
    tmp['period'] = period
    tmp['product'] = TARGET_PROD
    tmp['daynight'] = TARGET_TIME
    tmp['target_method'] = TARGET_COL
    tmp['source_file'] = fname
    rows.append(tmp[['UrbanId','year','period',TARGET_COL,'product','daynight','target_method','source_file']])
source_panel = pd.concat(rows, ignore_index=True)
monthly = source_panel[source_panel['period'].isin([6,7,8])].copy()
seasonal = source_panel[source_panel['period'].eq(TARGET_PERIOD)].copy()

phase2_raw = seasonal[['UrbanId','year',TARGET_COL,'product','daynight','period','target_method','source_file']].copy()
phase2_raw.to_csv(PHASE2_ROOT / 'phase2_primary_target_raw.csv', index=False)
print(f'Loaded selected files: {len(selected_files)}; expected files for full comparison: {len(expected_seasonal_pairs | expected_monthly_pairs)}')
print(f'Missing seasonal target files: {len(missing_seasonal_pairs)}; missing monthly comparison files: {len(missing_monthly_pairs)}')
print(f'Monthly records: {len(monthly):,}; JJA records: {len(seasonal):,}')
print(f'Saved: {PHASE2_ROOT / "phase2_primary_target_raw.csv"}')

Loaded selected files: 80; expected files for full comparison: 80
Missing seasonal target files: 0; missing monthly comparison files: 0
Monthly records: 611,760; JJA records: 203,920
Saved: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/outputs/phase2_outputs/phase2_primary_target_raw.csv


## Step 3 — Compare JJA seasonal target with monthly values

`aggfunc='first'` or “keep first row” is not used here. Duplicate source keys are explicitly marked as ambiguous and are not included in the comparison. Monthly average is calculated only for pairs where all three months are valid.

In [ ]:
# Composite validation is scoped to the cities mapped for this study (including disputed IDs for diagnosis),
# not every global city in the source archive. The precomputed period-22 value remains the primary target.
study_ids = set(pd.to_numeric(manifest['UrbanId'], errors='raise').astype(int))
monthly = monthly[monthly['UrbanId'].isin(study_ids)].copy()
seasonal = seasonal[seasonal['UrbanId'].isin(study_ids)].copy()

season_counts = seasonal.groupby(['UrbanId','year'], dropna=False).size().rename('n_season_rows').reset_index()
season_dup_keys = season_counts[season_counts.n_season_rows > 1][['UrbanId','year']].copy()
season_dup_keys['key_type'] = 'duplicate_seasonal_key'
season_dup_keys['n_rows'] = season_counts[season_counts.n_season_rows > 1]['n_season_rows'].values

month_counts = monthly.groupby(['UrbanId','year','period'], dropna=False).size().rename('n_rows').reset_index()
month_dup = month_counts[month_counts.n_rows > 1].copy()
month_dup['key_type'] = 'duplicate_month_key'
month_dup = month_dup.rename(columns={'n_rows':'n_rows_duplicate_key'})
month_dup['source_period'] = month_dup['period']
monthly_dup_key_set = set(map(tuple, month_dup[['UrbanId','year','period']].to_numpy()))
monthly['join_key'] = list(zip(monthly['UrbanId'], monthly['year'], monthly['period']))
monthly_unique = monthly[~monthly['join_key'].isin(monthly_dup_key_set)].drop(columns='join_key').copy()
season_dup_set = set(map(tuple, season_dup_keys[['UrbanId','year']].to_numpy()))
season_unique = seasonal[~seasonal.set_index(['UrbanId','year']).index.isin(season_dup_set)].copy()

if not monthly_unique.empty:
    monthly_wide = monthly_unique.pivot(index=['UrbanId','year'], columns='period', values=TARGET_COL).reset_index()
else:
    monthly_wide = pd.DataFrame(columns=['UrbanId','year',6,7,8])
monthly_wide.columns.name = None
for period in [6,7,8]:
    if period not in monthly_wide.columns:
        monthly_wide[period] = np.nan
monthly_wide = monthly_wide.rename(columns={6:'Jun_MEA',7:'Jul_MEA',8:'Aug_MEA'})

validation_unique = season_unique[['UrbanId','year',TARGET_COL,'source_file']].rename(
    columns={TARGET_COL:'season_MEA_period22','source_file':'season_source_file'}).merge(
        monthly_wide[['UrbanId','year','Jun_MEA','Jul_MEA','Aug_MEA']],
        on=['UrbanId','year'], how='outer', validate='one_to_one')
all_keys = pd.concat([seasonal[['UrbanId','year']], monthly[['UrbanId','year']],], ignore_index=True).drop_duplicates()
all_season_keys = all_keys.merge(season_counts, on=['UrbanId','year'], how='left', validate='one_to_one')
all_season_keys['n_season_rows'] = all_season_keys['n_season_rows'].fillna(0).astype(int)
all_season_keys = all_season_keys.merge(
    validation_unique[['UrbanId','year','season_MEA_period22','season_source_file','Jun_MEA','Jul_MEA','Aug_MEA']],
    on=['UrbanId','year'], how='left', validate='one_to_one')
monthly_city_year_keys = monthly[['UrbanId','year']].drop_duplicates()
monthly_city_year_set = set(map(tuple, monthly_city_year_keys.to_numpy()))
all_season_keys['has_seasonal_row'] = all_season_keys['n_season_rows'].gt(0)
all_season_keys['has_monthly_row'] = [((u,y) in monthly_city_year_set) for u,y in zip(all_season_keys.UrbanId,all_season_keys.year)]
all_season_keys['ambiguous_month_key'] = False
monthly_dup_city_years = set(map(tuple, month_dup[['UrbanId','year']].drop_duplicates().to_numpy())) if len(month_dup) else set()
all_season_keys['ambiguous_month_key'] = [((u,y) in monthly_dup_city_years) for u,y in zip(all_season_keys.UrbanId, all_season_keys.year)]
all_season_keys['has_all_three_months'] = all_season_keys[['Jun_MEA','Jul_MEA','Aug_MEA']].notna().all(axis=1)
all_season_keys['monthly_mean_MEA'] = all_season_keys[['Jun_MEA','Jul_MEA','Aug_MEA']].mean(axis=1).where(all_season_keys['has_all_three_months'])
all_season_keys['diff'] = all_season_keys['season_MEA_period22'] - all_season_keys['monthly_mean_MEA']
all_season_keys['abs_diff'] = all_season_keys['diff'].abs()
all_season_keys['comparison_status'] = 'READY_FOR_COMPARISON'
all_season_keys.loc[all_season_keys['n_season_rows'] == 0, 'comparison_status'] = 'MISSING_SEASONAL_ROW'
all_season_keys.loc[all_season_keys['n_season_rows'] > 1, 'comparison_status'] = 'AMBIGUOUS_SEASONAL_KEY'
all_season_keys.loc[all_season_keys['ambiguous_month_key'] & (all_season_keys['n_season_rows'] <= 1), 'comparison_status'] = 'AMBIGUOUS_MONTH_KEY'
all_season_keys.loc[all_season_keys['season_MEA_period22'].isna() & (all_season_keys['n_season_rows'] == 1), 'comparison_status'] = 'MISSING_SEASONAL_TARGET'
all_season_keys.loc[~all_season_keys['has_all_three_months'] & all_season_keys['comparison_status'].eq('READY_FOR_COMPARISON'), 'comparison_status'] = 'MISSING_MONTH'
all_season_keys = all_season_keys.merge(
    manifest[['UrbanId','Country','eligibility_status','disputed_id_flag']], on='UrbanId', how='left', validate='many_to_one')

ready = all_season_keys['comparison_status'].eq('READY_FOR_COMPARISON')
valid_resids = all_season_keys.loc[ready, 'abs_diff'].dropna()
HIGH_RESIDUAL_THRESHOLD_C = 1.0  # review-screen only; never an automatic exclusion threshold
high_residual_review = all_season_keys[ready & all_season_keys['abs_diff'].ge(HIGH_RESIDUAL_THRESHOLD_C)].copy()
high_residual_path = PHASE2_ROOT / 'task3_composite_high_residual_review.csv'
high_residual_review.to_csv(high_residual_path, index=False)

# Maintain a separate, editable disposition log for large residuals. Existing human decisions are preserved.
review_decisions_path = PHASE2_ROOT / 'task3_composite_review_decisions.csv'
review_key_cols = ['UrbanId','year']
review_value_cols = ['decision','explanation']
if review_decisions_path.exists():
    prior_decisions = pd.read_csv(review_decisions_path)
    if not set(review_key_cols + review_value_cols).issubset(prior_decisions.columns):
        raise ValueError(f'{review_decisions_path} must contain columns {review_key_cols + review_value_cols}')
    if prior_decisions.duplicated(review_key_cols).any():
        raise ValueError(f'{review_decisions_path} has duplicate UrbanId-year decisions.')
else:
    prior_decisions = pd.DataFrame(columns=review_key_cols + review_value_cols)
current_review = high_residual_review[[
    'UrbanId','year','Country','season_MEA_period22','Jun_MEA','Jul_MEA','Aug_MEA','monthly_mean_MEA','diff','abs_diff'
]].copy()
current_review = current_review.merge(prior_decisions[review_key_cols + review_value_cols], on=review_key_cols, how='left', validate='one_to_one')
# A default protocol disposition prevents a blank review log. It retains the precomputed JJA label,
# but explicitly does NOT claim the exact physical/source reason for a large monthly residual is known.
needs_disposition = current_review['decision'].isna() | current_review['decision'].astype('string').str.strip().eq('')
current_review.loc[needs_disposition, 'decision'] = 'PROTOCOL_DEFAULT_RETAIN_FLAGGED_FOR_SENSITIVITY'
current_review.loc[needs_disposition, 'explanation'] = (
    'Retain the precomputed period-22 JJA target as the primary label under the study protocol. '
    'The arithmetic mean of monthly SUHII is diagnostic and is not assumed to be numerically identical. '
    'This disposition does not prove the exact physical/source cause of the residual; preserve the flag '
    'and report this city-year in sensitivity/limitations discussion. Do not silently delete or replace it.'
)
current_keys = set(map(tuple, current_review[review_key_cols].to_numpy()))
if len(prior_decisions):
    old_extra = prior_decisions[~prior_decisions.apply(lambda r: (r['UrbanId'], r['year']) in current_keys, axis=1)].copy()
    if not old_extra.empty:
        old_extra['Country'] = np.nan
        for c in ['season_MEA_period22','Jun_MEA','Jul_MEA','Aug_MEA','monthly_mean_MEA','diff','abs_diff']:
            old_extra[c] = np.nan
        current_review = pd.concat([current_review, old_extra[current_review.columns]], ignore_index=True)
current_review.to_csv(review_decisions_path, index=False)
if high_residual_review.empty:
    high_residuals_logged = high_residual_path.exists()
else:
    current_key_set = set(map(tuple, current_review[review_key_cols].to_numpy()))
    expected_high_keys = set(map(tuple, high_residual_review[review_key_cols].to_numpy()))
    high_residuals_logged = high_residual_path.exists() and expected_high_keys.issubset(current_key_set)

summary = pd.DataFrame([{
    'run_id': RUN_ID,
    'validation_scope_city_ids': len(study_ids),
    'n_seasonal_rows': len(seasonal),
    'n_seasonal_unique_keys': int(seasonal[['UrbanId','year']].drop_duplicates().shape[0]),
    'n_duplicate_seasonal_keys': int(len(season_dup_keys)),
    'n_duplicate_month_keys': int(len(month_dup)),
    'n_strict_keys_compared': int(valid_resids.shape[0]),
    'n_non_comparable_city_years': int((~ready).sum()),
    'n_high_residuals_abs_diff_ge_1C': int(len(high_residual_review)),
    'high_residuals_logged_for_review': high_residuals_logged,
    'mean_abs_diff_C': float(valid_resids.mean()) if len(valid_resids) else np.nan,
    'median_abs_diff_C': float(valid_resids.median()) if len(valid_resids) else np.nan,
    'median_signed_diff_C': float(all_season_keys.loc[ready,'diff'].median()) if ready.any() else np.nan,
    'max_abs_diff_C': float(valid_resids.max()) if len(valid_resids) else np.nan,
    'precomputed_jja_target_rule_approved_by_protocol': JJA_TARGET_RULE_APPROVED_BY_PROTOCOL,
}])
validation_path = PHASE2_ROOT / 'task3_composite_validation.csv'
summary_path = PHASE2_ROOT / 'task3_composite_validation_summary.csv'
ambiguity_path = PHASE2_ROOT / 'task3_duplicate_or_ambiguous_keys.csv'
report_path = PHASE2_ROOT / 'task3_composite_validation_report.md'
all_season_keys.to_csv(validation_path, index=False)
summary.to_csv(summary_path, index=False)
ambig_parts = []
if len(season_dup_keys): ambig_parts.append(season_dup_keys)
if len(month_dup):
    ambig_parts.append(month_dup[['UrbanId','year','source_period','key_type','n_rows_duplicate_key']].rename(columns={'source_period':'period','n_rows_duplicate_key':'n_rows'}))
ambig = pd.concat(ambig_parts, ignore_index=True, sort=False) if ambig_parts else pd.DataFrame(columns=['UrbanId','year','period','key_type','n_rows'])
ambig.to_csv(ambiguity_path, index=False)
country_summary = all_season_keys.groupby(['Country','comparison_status'], dropna=False).agg(
    city_year_keys=('UrbanId','size'), unique_cities=('UrbanId','nunique')
).reset_index()
country_summary.to_csv(PHASE2_ROOT / 'task3_composite_validation_country_summary.csv', index=False)
s = summary.iloc[0]
report = f'''# JJA Composite Validation Report

- Run ID: `{RUN_ID}`
- Target: `{TARGET_PROD}` / `{TARGET_TIME}` / `{TARGET_COL}` / period `{TARGET_PERIOD}`
- Scope: only the study city mapping (including disputed IDs for diagnosis), not the full global city archive.
- Study city IDs in manifest: {len(study_ids):,}
- Seasonal rows for mapped cities: {int(s.n_seasonal_rows):,}
- Unique seasonal UrbanId-year keys: {int(s.n_seasonal_unique_keys):,}
- Duplicate seasonal keys: {int(s.n_duplicate_seasonal_keys):,}
- Duplicate monthly UrbanId-year-period keys: {int(s.n_duplicate_month_keys):,}
- Strict, unambiguous comparisons: {int(s.n_strict_keys_compared):,}
- Non-comparable city-years: {int(s.n_non_comparable_city_years):,}
- Absolute residuals >= {HIGH_RESIDUAL_THRESHOLD_C:.1f} C requiring explicit review: {int(s.n_high_residuals_abs_diff_ge_1C):,}
- Mean absolute difference: {s.mean_abs_diff_C if pd.notna(s.mean_abs_diff_C) else 'N/A'} C
- Median absolute difference: {s.median_abs_diff_C if pd.notna(s.median_abs_diff_C) else 'N/A'} C
- Median signed difference (seasonal minus monthly mean): {s.median_signed_diff_C if pd.notna(s.median_signed_diff_C) else 'N/A'} C
- Maximum absolute difference: {s.max_abs_diff_C if pd.notna(s.max_abs_diff_C) else 'N/A'} C
- High residuals written to the review-only output: {high_residuals_logged}
- Target rule: precomputed period-22 JJA remains the primary label; monthly arithmetic mean is diagnostic only.

**Interpretation:** The primary target is the source's precomputed period-22 JJA product. An unweighted mean of monthly SUHII is a useful diagnostic but is not required to equal the seasonal product exactly because seasonal compositing and valid observations may differ. Residuals at or above the screening threshold are retained in the target and exported for transparency; they are not automatic exclusions. Only separate evidence of a wrong source key, product, or corrupted value justifies excluding a city-year. Each high residual receives an explicit protocol disposition in `task3_composite_review_decisions.csv`. The default is to retain the precomputed target, preserve the flag, and disclose the uncertainty; it is not a claim that the underlying physical/source cause has been independently proven.

**Status:** {'DIAGNOSTIC COMPLETE — high residuals flagged and retained under the precomputed-target rule' if len(high_residual_review) else 'DIAGNOSTIC COMPLETE — no residuals met the high-residual review threshold'}.
'''
report_path.write_text(report, encoding='utf-8')
print(summary.to_string(index=False))
print(f'Saved: {validation_path}\nSaved: {summary_path}\nSaved: {ambiguity_path}\nSaved: {report_path}\nHigh residual review: {high_residual_path}\nDisposition log: {review_decisions_path}')

         run_id  validation_scope_city_ids  n_seasonal_rows  n_seasonal_unique_keys  n_duplicate_seasonal_keys  n_duplicate_month_keys  n_strict_keys_compared  n_non_comparable_city_years  n_high_residuals_abs_diff_ge_1C  high_residuals_logged_for_review  mean_abs_diff_C  median_abs_diff_C  median_signed_diff_C  max_abs_diff_C  precomputed_jja_target_rule_approved_by_protocol
20261018_002711                       1634            32680                   32680                          0                       0                   32476                          204                               19                              True         0.007467           0.000989             -0.000007        3.942026                                              True
Saved: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/outputs/phase2_outputs/task3_composite_validation.csv
Saved: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/outputs/phase2_o

## Step 4 — Annual Coverage and Quarantine Audit

This step counts the expected cities in the manifest, the observed keys in the file, and the actual valid target for each country-year. Differences are calculated using the same code; the theoretical full panel and the observed panel are not confused.

In [ ]:
manifest_small = manifest[['UrbanId','Country','eligibility_status','disputed_id_flag']].copy()
if manifest_small['UrbanId'].duplicated().any():
    raise ValueError('Manifest must have one row per UrbanId.')

raw_joined = phase2_raw.merge(manifest_small, on='UrbanId', how='left', validate='many_to_one')
raw_joined['Country'] = raw_joined['Country'].fillna('UNKNOWN')
raw_joined['eligibility_status'] = raw_joined['eligibility_status'].fillna('UNMAPPED')

coverage_rows = []
for country in ['China','Türkiye']:
    eligible_ids = set(manifest.loc[(manifest.Country == country) & manifest.eligibility_status.eq('INCLUDED'),'UrbanId'])
    for year in YEARS:
        sub = raw_joined[(raw_joined['Country'] == country) & (raw_joined['eligibility_status'] == 'INCLUDED') & (raw_joined['year'] == year)]
        key_counts = sub.groupby(['UrbanId','year']).size()
        dup_keys = int((key_counts > 1).sum())
        observed_keys = set(sub['UrbanId'].unique())
        valid_target_keys = set(sub.loc[sub[TARGET_COL].notna(), 'UrbanId'].unique())
        coverage_rows.append({
            'year': year, 'country': country, 'method': TARGET_COL, 'period': TARGET_PERIOD,
            'eligible_cities_in_manifest': len(eligible_ids),
            'observed_city_year_keys': len(observed_keys),
            'valid_target_city_years': len(valid_target_keys),
            'missing_target_city_years_among_observed': len(observed_keys - valid_target_keys),
            'missing_city_year_records': max(0, len(eligible_ids) - len(observed_keys)),
            'duplicate_city_year_keys': dup_keys,
            'complete_expected_panel': len(observed_keys) == len(eligible_ids) and len(valid_target_keys) == len(eligible_ids) and dup_keys == 0,
        })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(PHASE2_ROOT / 'task4_year_country_coverage.csv', index=False)

quar = raw_joined[raw_joined['eligibility_status'].ne('INCLUDED')].copy()
quar_summary = quar.groupby(['year','Country','eligibility_status'], dropna=False).agg(
    rows=('UrbanId','size'), unique_urbanids=('UrbanId','nunique'), missing_target_rows=(TARGET_COL, lambda s: int(s.isna().sum()))
).reset_index()
quar_summary.to_csv(PHASE2_ROOT / 'task4_quarantined_summary.csv', index=False)
missing_years = coverage[coverage['missing_city_year_records'] > 0].copy()
missing_years.to_csv(PHASE2_ROOT / 'task4_missing_city_years.csv', index=False)

summary_text = [
    '# Task 4 — Year/Country Coverage Summary', '',
    f'- Run ID: `{RUN_ID}`', f'- Year range: {YEAR_MIN}–{YEAR_MAX}',
    f'- Eligible manifest cities: {len(eligible_manifest):,}',
    f'- Theoretical full panel (eligible cities × 20 years): {len(eligible_manifest) * len(YEARS):,}',
    f'- Target rows loaded (all global source IDs): {len(phase2_raw):,}', '',
    'Coverage table is the source of truth. `observed_city_year_keys`, `valid_target_city_years`, missing targets, missing city-year records, and duplicate keys are separated intentionally.', '',
    coverage.to_markdown(index=False), ''
]
(PHASE2_ROOT / 'task4_coverage_summary.md').write_text('\n'.join(summary_text), encoding='utf-8')
print(coverage.groupby('country')[['eligible_cities_in_manifest','observed_city_year_keys','valid_target_city_years','missing_city_year_records','duplicate_city_year_keys']].sum())
print('Saved coverage, quarantined summary, missing city-years, and Markdown report.')

         eligible_cities_in_manifest  observed_city_year_keys  \
country                                                         
China                          28820                    28820   
Türkiye                         3600                     3600   

         valid_target_city_years  missing_city_year_records  \
country                                                       
China                      28820                          0   
Türkiye                     3600                          0   

         duplicate_city_year_keys  
country                            
China                           0  
Türkiye                         0  
Saved coverage, quarantined summary, missing city-years, and Markdown report.


## Step 5 — Clean target table and actual exclusion log

A reason is written for each row that is quarantined, out of scope, missing target, or has an unclear key. The entire duplicate `UrbanId-year` group is quarantined; a row is not selected simply because it came one row earlier.

In [ ]:
cleaning = raw_joined.copy()
cleaning['reason_code'] = ''
cleaning['reason_detail'] = ''

m = cleaning['eligibility_status'].eq('UNMAPPED')
cleaning.loc[m, ['reason_code','reason_detail']] = ['OUT_OF_SCOPE_CITY','UrbanId is not in the approved China/Türkiye city manifest']
m = cleaning['reason_code'].eq('') & cleaning['eligibility_status'].eq('QUARANTINED')
cleaning.loc[m, ['reason_code','reason_detail']] = ['QUARANTINED_CITY','City manifest marks this UrbanId as quarantined']
m = cleaning['reason_code'].eq('') & ~cleaning['Country'].isin(['China','Türkiye'])
cleaning.loc[m, ['reason_code','reason_detail']] = ['OUT_OF_SCOPE_COUNTRY','Country is not in the study scope']

# A duplicate key is ambiguous even if one of its rows has a missing target.
key_scope = cleaning[cleaning['reason_code'].eq('')]
key_sizes = key_scope.groupby(['UrbanId','year']).size()
dup_key_tuples = set(key_sizes[key_sizes > 1].index.tolist())
row_keys = pd.MultiIndex.from_frame(cleaning[['UrbanId','year']])
dup_key_index = pd.MultiIndex.from_tuples(list(dup_key_tuples), names=['UrbanId','year']) if dup_key_tuples else pd.MultiIndex.from_arrays([[],[]], names=['UrbanId','year'])
dup_row_mask = row_keys.isin(dup_key_index) & cleaning['reason_code'].eq('').to_numpy()
cleaning.loc[dup_row_mask, ['reason_code','reason_detail']] = ['AMBIGUOUS_DUPLICATE_KEY','More than one source row exists for this UrbanId-year; all rows quarantined rather than keeping an arbitrary row']

# Apply only explicit, documented decisions from the high-residual review log.
review_path = PHASE2_ROOT / 'task3_composite_review_decisions.csv'
if review_path.exists():
    review_decisions = pd.read_csv(review_path)
    if not {'UrbanId','year','decision','explanation'}.issubset(review_decisions.columns):
        raise ValueError(f'{review_path} must contain UrbanId, year, decision, explanation')
    if review_decisions.duplicated(['UrbanId','year']).any():
        raise ValueError(f'{review_path} contains duplicate city-year review decisions')
    populated_decisions = review_decisions['decision'].fillna('').astype(str).str.strip().ne('')
    valid_labels = {'ACCEPT_WITH_EXPLANATION','EXCLUDE_CITY_YEAR','PROTOCOL_DEFAULT_RETAIN_FLAGGED_FOR_SENSITIVITY'}
    invalid_labels = set(review_decisions.loc[populated_decisions, 'decision'].astype(str).str.strip()) - valid_labels
    if invalid_labels:
        raise ValueError(f'{review_path} has unsupported decisions: {sorted(invalid_labels)}')
    missing_rationale = populated_decisions & review_decisions['explanation'].fillna('').astype(str).str.strip().eq('')
    if missing_rationale.any():
        raise ValueError(f'{review_path} contains a recorded decision without an explanation.')
    excluded_review_keys = set(map(tuple, review_decisions.loc[review_decisions['decision'].eq('EXCLUDE_CITY_YEAR'), ['UrbanId','year']].to_numpy()))
    cleaning_keys = list(zip(cleaning['UrbanId'], cleaning['year']))
    review_exclude_mask = cleaning['reason_code'].eq('') & pd.Series([k in excluded_review_keys for k in cleaning_keys], index=cleaning.index)
    cleaning.loc[review_exclude_mask, ['reason_code','reason_detail']] = ['COMPOSITE_REVIEW_EXCLUDED','Explicit EXCLUDE_CITY_YEAR disposition in task3_composite_review_decisions.csv']

m = cleaning['reason_code'].eq('') & cleaning[TARGET_COL].isna()
cleaning.loc[m, ['reason_code','reason_detail']] = ['MISSING_TARGET','Intensity_MEA is missing in the precomputed JJA source; no imputation applied']

excluded = cleaning[cleaning['reason_code'].ne('')].copy()
excluded['action'] = 'REMOVED_FROM_PRIMARY_TARGET'
excluded['kept_or_removed'] = 'REMOVED'
excluded['run_id'] = RUN_ID
exclude_cols = ['UrbanId','year','Country','period','product','daynight','target_method','source_file','reason_code','reason_detail','action','kept_or_removed','run_id']
excluded[exclude_cols].to_csv(PHASE2_ROOT / 'task5_cleaning_log.csv', index=False)

final = cleaning[cleaning['reason_code'].eq('')].copy()
final = final[['UrbanId','year',TARGET_COL,'Country','source_file']].sort_values(['UrbanId','year']).reset_index(drop=True)
if final.duplicated(['UrbanId','year']).any():
    raise AssertionError('Internal error: final target still has duplicate UrbanId-year keys.')
if final[TARGET_COL].isna().any():
    raise AssertionError('Internal error: final target still has missing labels.')
if not final['year'].between(YEAR_MIN,YEAR_MAX).all():
    raise AssertionError('Internal error: out-of-range year remains.')
if not final['Country'].isin(['China','Türkiye']).all():
    raise AssertionError('Internal error: out-of-scope country remains.')

final_path = PHASE2_ROOT / 'phase2_cleaned_target.csv'
final.to_csv(final_path, index=False)
country_year_summary = final.groupby(['Country','year']).agg(
    n_unique_cities=('UrbanId','nunique'), n_records=('UrbanId','size'),
    target_mean=(TARGET_COL,'mean'), target_std=(TARGET_COL,'std'),
    target_min=(TARGET_COL,'min'), target_max=(TARGET_COL,'max')
).reset_index()
country_year_summary.to_csv(PHASE2_ROOT / 'task5_country_year_summary.csv', index=False)
country_summary = final.groupby('Country').agg(
    cities=('UrbanId','nunique'), rows=('UrbanId','size'), mean=(TARGET_COL,'mean'),
    std=(TARGET_COL,'std'), min=(TARGET_COL,'min'), max=(TARGET_COL,'max')
).reset_index()

# IQR is an audit flag only; never remove records solely because they are statistical outliers.
outlier_review = final[['UrbanId','year','Country',TARGET_COL]].copy()
outlier_review['q1_country'] = outlier_review.groupby('Country')[TARGET_COL].transform(lambda s: s.quantile(0.25))
outlier_review['q3_country'] = outlier_review.groupby('Country')[TARGET_COL].transform(lambda s: s.quantile(0.75))
outlier_review['iqr_country'] = outlier_review['q3_country'] - outlier_review['q1_country']
outlier_review['lower_iqr_bound'] = outlier_review['q1_country'] - 1.5*outlier_review['iqr_country']
outlier_review['upper_iqr_bound'] = outlier_review['q3_country'] + 1.5*outlier_review['iqr_country']
outlier_review['iqr_outlier_candidate'] = (outlier_review[TARGET_COL] < outlier_review['lower_iqr_bound']) | (outlier_review[TARGET_COL] > outlier_review['upper_iqr_bound'])
outlier_review['action'] = 'REVIEW_ONLY_NOT_REMOVED'
outlier_review.to_csv(PHASE2_ROOT / 'task5_outlier_review.csv', index=False)

validation = pd.DataFrame([{
    'run_id': RUN_ID, 'starting_rows': len(cleaning), 'final_rows': len(final),
    'unique_urbanids': int(final['UrbanId'].nunique()),
    'n_unique_keys_in_final': int(final[['UrbanId','year']].drop_duplicates().shape[0]),
    'duplicate_rows_in_final': int(final.duplicated(['UrbanId','year']).sum()),
    'missing_targets_in_final': int(final[TARGET_COL].isna().sum()),
    'out_of_range_year_rows': int((~final['year'].between(YEAR_MIN,YEAR_MAX)).sum()),
    'out_of_scope_country_rows': int((~final['Country'].isin(['China','Türkiye'])).sum()),
    'excluded_rows_logged': len(excluded),
    'quarantined_city_year_rows': int(excluded['reason_code'].eq('QUARANTINED_CITY').sum()),
    'unknown_city_rows': int(excluded['reason_code'].eq('OUT_OF_SCOPE_CITY').sum()),
    'ambiguous_duplicate_key_rows': int(excluded['reason_code'].eq('AMBIGUOUS_DUPLICATE_KEY').sum()),
    'no_duplicate_keys_in_final': not final.duplicated(['UrbanId','year']).any(),
    'all_targets_present': not final[TARGET_COL].isna().any(),
    'all_years_in_range': bool(final['year'].between(YEAR_MIN,YEAR_MAX).all()),
    'all_country_in_scope': bool(final['Country'].isin(['China','Türkiye']).all()),
}])
validation.to_csv(PHASE2_ROOT / 'task5_target_validation.csv', index=False)

clean_report = [
    '# Task 5 — Target Cleaning Report','',f'- Run ID: `{RUN_ID}`',
    f'- Input seasonal rows: {len(cleaning):,}',f'- Final clean target rows: {len(final):,}',
    f'- Excluded rows logged: {len(excluded):,}',
    f'- Unique city-years in final: {final[["UrbanId","year"]].drop_duplicates().shape[0]:,}',
    f'- Duplicate target keys in final: {int(final.duplicated(["UrbanId","year"]).sum())}', '',
    '## Country summary','',country_summary.to_markdown(index=False),'',
    '## Exclusion counts','',excluded['reason_code'].value_counts(dropna=False).to_frame('rows').to_markdown(),''
]
(PHASE2_ROOT / 'task5_cleaning_report.md').write_text('\n'.join(clean_report), encoding='utf-8')
print('Final target:', final.shape)
print(country_summary.to_string(index=False))
print('Exclusions:', excluded['reason_code'].value_counts().to_dict())
print('Saved cleaned target + logs + summaries + validation report.')

Final target: (32420, 5)
Country  cities  rows     mean      std       min      max
  China    1441 28820 0.810849 0.503295 -3.014442 4.082289
Türkiye     180  3600 0.962272 0.701136 -1.773750 4.219071
Exclusions: {'OUT_OF_SCOPE_CITY': 171240, 'QUARANTINED_CITY': 260}
Saved cleaned target + logs + summaries + validation report.


## Step 6 — Basic Geographic Features and Leakage Control

The main model only uses geographic/temporal features such as year, coordinates, and area. Country and `UrbanId` are for splitting and mapping only; they are not model inputs. Monthly MEA targets or derivatives of targets cannot be included in the feature list.

In [ ]:
coords = manifest.loc[manifest.eligibility_status.eq('INCLUDED'), ['UrbanId','Longitude','Latitude','Area']].copy()
if coords['UrbanId'].duplicated().any():
    raise ValueError('Eligible manifest has duplicate UrbanId values.')
coords[['Longitude','Latitude','Area']] = coords[['Longitude','Latitude','Area']].apply(pd.to_numeric, errors='coerce')
features = final.merge(coords, on='UrbanId', how='left', validate='many_to_one')
if len(features) != len(final):
    raise AssertionError('Metadata merge unexpectedly changed row count.')
if features[['Longitude','Latitude','Area']].isna().any().any():
    n_missing = int(features[['Longitude','Latitude','Area']].isna().any(axis=1).sum())
    raise ValueError(f'{n_missing} target rows are missing required geographic metadata. Inspect manifest before modelling.')
if (features['Area'] <= 0).any():
    raise ValueError('Non-positive Area detected; do not silently log-transform invalid area values.')

# Avoid redundant deterministic predictors: in this north-hemisphere study abs(Latitude)=Latitude,
# and including both Area and log(Area) duplicates information. Keep a compact, interpretable baseline.
features['area_log'] = np.log(features['Area'])
BASE_FEATURES = ['year','Longitude','Latitude','area_log']
TARGET_NAME = TARGET_COL
FORBIDDEN_FEATURES = {'UrbanId','Country','country','Intensity_EA','Intensity_IEA','Intensity_MEA','Intensity_DEA',
                      'Intensity_MEA_jun','Intensity_MEA_jul','Intensity_MEA_aug','Jun_MEA','Jul_MEA','Aug_MEA'}
if set(BASE_FEATURES) & FORBIDDEN_FEATURES:
    raise AssertionError(f'Forbidden predictor leakage: {set(BASE_FEATURES) & FORBIDDEN_FEATURES}')

feature_path = PHASE2_ROOT / 'task6_feature_matrix.csv'
features.to_csv(feature_path, index=False)
leakage_rows = []
for f in BASE_FEATURES:
    leakage_rows.append({'feature':f,'in_model_feature_set':True,'missing_values':int(features[f].isna().sum()),
                         'source':'clean target + approved city metadata/derived area transform',
                         'target_derived_feature':False,'included_in_model':True})
for forbidden in sorted(FORBIDDEN_FEATURES):
    leakage_rows.append({'feature':forbidden,'in_model_feature_set':False,'missing_values':np.nan,
                         'source':'forbidden/identifier/target-derived if present','target_derived_feature':True,'included_in_model':False})
pd.DataFrame(leakage_rows).to_csv(PHASE2_ROOT / 'task6_leakage_audit.csv', index=False)
coverage_rows = []
for country, sub in features.groupby('Country'):
    for f in BASE_FEATURES:
        coverage_rows.append({'Country':country,'feature':f,'n_rows':len(sub),'n_missing':int(sub[f].isna().sum()),
                              'missing_pct':100*sub[f].isna().mean()})
pd.DataFrame(coverage_rows).to_csv(PHASE2_ROOT / 'task6_feature_coverage_by_country.csv', index=False)

# VIF from the predictor matrix is an audit signal, not an automatic feature deletion rule.
Xv = features[BASE_FEATURES].astype(float).replace([np.inf,-np.inf], np.nan).dropna()
vif_rows=[]
for feature in BASE_FEATURES:
    y = Xv[feature].to_numpy(dtype=float)
    others = Xv.drop(columns=[feature]).to_numpy(dtype=float)
    design = np.column_stack([np.ones(len(others)), others])
    fitted = design @ np.linalg.lstsq(design, y, rcond=None)[0]
    sst = float(np.sum((y-y.mean())**2))
    sse = float(np.sum((y-fitted)**2))
    r2_feature = 1.0 - sse/sst if sst > 0 else 1.0
    denominator = max(0.0, 1.0-r2_feature)
    vif_value = float(1.0/denominator) if denominator > 1e-12 else float('inf')
    vif_rows.append({'feature':feature,'vif':vif_value,'n_rows':len(Xv),
                     'interpretation':'review if VIF > 5; diagnostic only, not an automatic exclusion'})
pd.DataFrame(vif_rows).to_csv(PHASE2_ROOT / 'task6_vif_log.csv', index=False)

feature_dict = [
    '# Task 6 — Baseline Feature Dictionary','',
    '| Feature | Meaning | Unit | Source | Model use |', '|---|---|---|---|---|',
    '| `year` | Target year | year | Selected seasonal target filename | Included |',
    '| `Longitude` | City reference longitude | degrees east | Approved city mapping | Included |',
    '| `Latitude` | City reference latitude | degrees north | Approved city mapping | Included |',
    '| `Area` | Original city-area metadata | source metadata unit; confirm documentation | Approved city mapping | Metadata only |',
    '| `area_log` | Natural logarithm of positive `Area` | log-unit | Derived from Area | Included |', '',
    'The baseline deliberately excludes `abs_latitude` (identical to Latitude for these mapped Northern Hemisphere cities) and excludes raw Area when `area_log` is used, to avoid redundant predictors.',
    'The external feature notebook uses point elevation and nearest-grid ERA5 values; the NDVI extraction is a point sample at the mapped city coordinate, not an area-weighted city-polygon mean.',
    '**Leakage rule:** `UrbanId`, country labels, monthly MEA target pieces, and the seasonal target itself are excluded from predictors. `Country` is used only to assign source/target roles.'
]
(PHASE2_ROOT / 'task6_feature_dictionary.md').write_text('\n'.join(feature_dict), encoding='utf-8')
print(f'Saved baseline feature matrix {features.shape}; predictors={BASE_FEATURES}; VIF/leakage/coverage reports created.')

Saved baseline feature matrix (32420, 9); predictors=['year', 'Longitude', 'Latitude', 'area_log']; VIF/leakage/coverage reports created.


## Step 7 — City-based, fixed-test baseline transfer experiment

First, 20% of the target country's cities are locked as a fixed test group. Few-shot cities are selected only from the adaptation pool remaining in the target country; different proportions are evaluated in the same test cities. Ridge and Random Forest run in both directions. Preprocessing is only within the `Pipeline` fitted to the training data.

This initial baseline run is not model selection/hyperparameter optimization; the test set is not changed based on performance results.

In [ ]:
fm = pd.read_csv(feature_path)
MODEL_FEATURES = BASE_FEATURES.copy()
if fm.duplicated(['UrbanId','year']).any():
    raise ValueError('Feature matrix has duplicate UrbanId-year keys.')
if not fm['Country'].isin(['China','Türkiye']).all():
    raise ValueError('Feature matrix contains out-of-scope country labels.')
if not set(MODEL_FEATURES + [TARGET_NAME,'UrbanId','Country']).issubset(fm.columns):
    raise ValueError(f'Feature matrix is missing required columns: {set(MODEL_FEATURES + [TARGET_NAME,"UrbanId","Country"]) - set(fm.columns)}')
fm_clean = fm.dropna(subset=MODEL_FEATURES + [TARGET_NAME,'Country']).copy()


def make_city_pool_split(frame, source_country, target_country, test_fraction=TEST_FRACTION, seed=TEST_LOCK_SEED):
    source_ids = sorted(frame.loc[frame.Country.eq(source_country),'UrbanId'].astype(int).unique().tolist())
    target_ids = sorted(frame.loc[frame.Country.eq(target_country),'UrbanId'].astype(int).unique().tolist())
    if len(source_ids) < 2 or len(target_ids) < 2:
        raise ValueError(f'Insufficient cities for {source_country}->{target_country}: source={len(source_ids)}, target={len(target_ids)}')
    rng = np.random.default_rng(seed + (1 if target_country == 'Türkiye' else 2))
    perm = rng.permutation(target_ids)
    n_test = max(1, int(round(len(target_ids) * test_fraction)))
    test_ids = sorted(map(int, perm[:n_test]))
    adapt_ids = sorted(set(target_ids) - set(test_ids))
    return source_ids, adapt_ids, test_ids


def build_model(model_name, seed):
    if model_name == 'source_mean':
        return DummyRegressor(strategy='mean')
    if model_name == 'ridge':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
            ('model', Ridge(alpha=1.0))
        ])
    if model_name == 'rf':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=seed, n_jobs=-1))
        ])
    raise ValueError(model_name)


def metrics_for_predictions(test_df, y_true, y_pred):
    residual = np.asarray(y_true) - np.asarray(y_pred)
    per_city = pd.DataFrame({'UrbanId':test_df['UrbanId'].to_numpy(), 'abs_err':np.abs(residual), 'sq_err':residual**2}).groupby('UrbanId').agg(city_mae=('abs_err','mean'), city_mse=('sq_err','mean'))
    return {
        'r2': float(r2_score(y_true,y_pred)) if len(y_true)>1 else np.nan,
        'rmse': float(np.sqrt(mean_squared_error(y_true,y_pred))),
        'mae': float(mean_absolute_error(y_true,y_pred)),
        'macro_city_mae': float(per_city['city_mae'].mean()),
        'macro_city_rmse': float(np.sqrt(per_city['city_mse'].mean())),
    }

results = []
split_rows = []
selection_rows = []
DIRECTIONS = [
    ('China_to_Turkiye','China','Türkiye'),
    ('Turkiye_to_China','Türkiye','China'),
]
MODEL_NAMES = ['source_mean','ridge','rf']
for direction, source_country, target_country in DIRECTIONS:
    source_ids, adapt_ids, test_ids = make_city_pool_split(fm_clean, source_country, target_country)
    split_rows.extend([{'direction':direction,'UrbanId':u,'country_role':'SOURCE_TRAIN','split_lock_seed':TEST_LOCK_SEED} for u in source_ids])
    split_rows.extend([{'direction':direction,'UrbanId':u,'country_role':'TARGET_ADAPTATION_POOL','split_lock_seed':TEST_LOCK_SEED} for u in adapt_ids])
    split_rows.extend([{'direction':direction,'UrbanId':u,'country_role':'TARGET_TEST_LOCKED','split_lock_seed':TEST_LOCK_SEED} for u in test_ids])
    if set(source_ids) & set(adapt_ids) or set(source_ids) & set(test_ids) or set(adapt_ids) & set(test_ids):
        raise AssertionError(f'City split overlap detected for {direction}')
    for ratio in FEW_SHOT_RATIOS:
        seed_list = [0] if ratio == 0 else ADAPTATION_SEEDS
        for shot_seed in seed_list:
            if ratio == 0:
                shot_ids = []
            else:
                rng = np.random.default_rng(11000 + shot_seed + (1 if direction == 'China_to_Turkiye' else 100))
                n_shot = min(len(adapt_ids), max(1, int(round(len(adapt_ids) * ratio))))
                shot_ids = sorted(map(int, rng.permutation(np.asarray(adapt_ids, dtype=int))[:n_shot]))
            selection_rows.extend([{'direction':direction,'ratio_of_target_adaptation_pool':ratio,'seed':shot_seed,'UrbanId':u} for u in shot_ids])
            train_ids = sorted(set(source_ids) | set(shot_ids))
            if set(train_ids) & set(test_ids):
                raise AssertionError('Target test city leakage detected.')
            train_df = fm_clean[fm_clean.UrbanId.isin(train_ids)].copy()
            test_df = fm_clean[fm_clean.UrbanId.isin(test_ids)].copy()
            if train_df.empty or test_df.empty:
                raise ValueError(f'Empty train/test subset for {direction}, ratio={ratio}, seed={shot_seed}')
            city_overlap = len(set(train_df['UrbanId'].astype(int).unique()) & set(test_df['UrbanId'].astype(int).unique()))
            if city_overlap != 0:
                raise AssertionError(f'Target test city overlap is {city_overlap} for {direction}, ratio={ratio}, seed={shot_seed}')
            for model_name in MODEL_NAMES:
                model = build_model(model_name, shot_seed)
                model.fit(train_df[MODEL_FEATURES], train_df[TARGET_NAME])
                pred = model.predict(test_df[MODEL_FEATURES])
                met = metrics_for_predictions(test_df, test_df[TARGET_NAME].to_numpy(), pred)
                results.append({
                    'direction':direction,'source_country':source_country,'target_country':target_country,
                    'model':model_name,'feature_set':'geo_baseline','few_shot_ratio':ratio,'seed':shot_seed,
                    'n_source_cities':len(source_ids),'n_adaptation_pool_cities':len(adapt_ids),
                    'n_shot_cities':len(shot_ids),'n_test_cities':len(test_ids),
                    'n_train_rows':len(train_df),'n_test_rows':len(test_df),'city_overlap':city_overlap,
                    **met
                })

results_df = pd.DataFrame(results)
if results_df.empty: raise RuntimeError('No baseline transfer results were generated.')
expected_settings = 1 + (len(FEW_SHOT_RATIOS)-1)*len(ADAPTATION_SEEDS)
expected_runs = len(DIRECTIONS) * len(MODEL_NAMES) * expected_settings
if len(results_df) != expected_runs:
    raise AssertionError(f'Expected {expected_runs} model evaluation rows, created {len(results_df)}.')
if set(results_df['model']) != set(MODEL_NAMES):
    raise AssertionError(f'Model result set incomplete: {set(results_df["model"])}')
results_path = PHASE2_ROOT / 'task7_transfer_results.csv'
results_df.to_csv(results_path, index=False)
pd.DataFrame(split_rows).to_csv(PHASE2_ROOT / 'task7_split_manifest.csv', index=False)
pd.DataFrame(selection_rows, columns=['direction','ratio_of_target_adaptation_pool','seed','UrbanId']).to_csv(PHASE2_ROOT / 'task7_few_shot_city_selections.csv', index=False)

split_report = [
    '# Task 7 — Baseline Transfer Split Report','',f'- Run ID: `{RUN_ID}`',
    f'- Test fraction per target country: {TEST_FRACTION:.0%}',
    f'- Locked test selection seed: {TEST_LOCK_SEED}',
    '- Test cities remain fixed across few-shot ratios within each direction; adaptation cities come only from the remaining target-country pool.',
    '- `few_shot_ratio` is the share of the target-country adaptation pool, not the whole target country.',
    '- Models: source-country mean baseline, Ridge, Random Forest. Fixed parameters are used; target test scores do not select models.',
    '- Preprocessing for Ridge/RF is fitted only on training data. The source mean baseline does not use predictor values.',
    '- Target country labels are used to construct splits, never as model predictors.',
    '- Row-weighted and city-macro metrics are reported. The same test city IDs are used across models and ratios in each direction.','',
    results_df.groupby(['direction','model','few_shot_ratio'])[['r2','rmse','mae','macro_city_mae','macro_city_rmse']].mean().reset_index().to_markdown(index=False),''
]
(PHASE2_ROOT / 'task7_partition_report.md').write_text('\n'.join(split_report), encoding='utf-8')
print(results_df.groupby(['direction','model','few_shot_ratio'])[['r2','rmse','mae','macro_city_mae']].mean().to_string())
print(f'Saved {len(results_df)} model-evaluation rows: {results_path}')

                                                   r2      rmse       mae  macro_city_mae
direction        model       few_shot_ratio                                              
China_to_Turkiye rf          0.00           -0.761272  1.121264  0.895382        0.895382
                             0.01           -0.648577  1.082443  0.870316        0.870316
                             0.05           -0.289663  0.957984  0.774618        0.774618
                             0.10           -0.260734  0.945302  0.759694        0.759694
                             0.20           -0.234613  0.935927  0.740602        0.740602
                             0.50           -0.026488  0.853156  0.688538        0.688538
                 ridge       0.00           -1.739504  1.398398  1.180408        1.180408
                             0.01           -1.479293  1.329926  1.113370        1.113370
                             0.05           -0.630796  1.077832  0.879635        0.879635
          

## Step 8 — Quality Gates and Decision Recording

The mere existence of files does not count as “GO.” This notebook automatically returns **NO-GO / REVIEW REQUIRED** until the composite verification has been human-reviewed and approved. This was intentionally set up this way; the old notebook's “16/16 files exist, therefore GO” approach was not sufficient evidence.

In [ ]:
# Reconcile observed non-missing target city-years to the final label table by key sets.
# Missing labels are not imputed; duplicate keys and explicit exclusions must be logged.
source_valid = raw_joined.loc[
    raw_joined['eligibility_status'].eq('INCLUDED')
    & raw_joined['Country'].isin(['China','Türkiye'])
    & raw_joined[TARGET_COL].notna(), ['Country','UrbanId','year']
].drop_duplicates()
source_valid_key_set = set(map(tuple, source_valid[['Country','UrbanId','year']].to_numpy()))
ambiguous_key_set = set(map(tuple, excluded.loc[excluded['reason_code'].eq('AMBIGUOUS_DUPLICATE_KEY'), ['Country','UrbanId','year']].drop_duplicates().to_numpy()))
review_exclusion_key_set = set(map(tuple, excluded.loc[excluded['reason_code'].eq('COMPOSITE_REVIEW_EXCLUDED'), ['Country','UrbanId','year']].drop_duplicates().to_numpy()))
coverage_reconciliation_rows = []
for country in ['China','Türkiye']:
    for year in YEARS:
        raw_valid_keys = {key for key in source_valid_key_set if key[0] == country and int(key[2]) == int(year)}
        duplicate_keys = raw_valid_keys & ambiguous_key_set
        review_keys = raw_valid_keys & review_exclusion_key_set
        expected_final = raw_valid_keys - duplicate_keys - review_keys
        final_keys = set(map(tuple, final.loc[(final['Country'].eq(country)) & (final['year'].eq(year)), ['Country','UrbanId','year']].drop_duplicates().to_numpy()))
        coverage_reconciliation_rows.append({
            'Country':country,'year':year,'raw_valid_target_city_years':len(raw_valid_keys),
            'duplicate_city_year_keys_quarantined':len(duplicate_keys),
            'explicit_review_exclusions':len(review_keys),'final_target_rows':len(final_keys),
            'expected_final_after_logged_actions':len(expected_final),
            'reconciles':expected_final == final_keys,
        })
coverage_reconciliation = pd.DataFrame(coverage_reconciliation_rows)
coverage_reconciliation.to_csv(PHASE2_ROOT/'task5_coverage_reconciliation.csv',index=False)
coverage_reconciles = bool(not coverage_reconciliation.empty and coverage_reconciliation['reconciles'].all())
coverage_keyshape_ok = (len(coverage)==len(['China','Türkiye'])*len(YEARS)
                        and not coverage.duplicated(['country','year']).any())

checks = {
    'manifest_unique_UrbanId': not manifest['UrbanId'].duplicated().any(),
    'country_year_coverage_table_complete': coverage_keyshape_ok,
    'final_target_rows_reconcile_to_logged_exclusions': coverage_reconciles,
    'disputed_list_has_13_ids': len(disputed_ids) == 13,
    'all_disputed_ids_quarantined_and_absent_from_final': (int(manifest['disputed_id_flag'].sum()) == 13 and not final['UrbanId'].isin(disputed_ids).any()),
    'all_expected_seasonal_files_present': len(missing_seasonal_pairs) == 0,
    'all_expected_monthly_files_present': len(missing_monthly_pairs) == 0,
    'primary_target_has_only_period_22': phase2_raw['period'].eq(22).all(),
    'primary_target_has_only_Mod2_Nig': phase2_raw['product'].eq('Mod2').all() and phase2_raw['daynight'].eq('Nig').all(),
    'clean_target_unique_UrbanId_year': not final.duplicated(['UrbanId','year']).any(),
    'clean_target_complete_labels': not final[TARGET_NAME].isna().any(),
    'clean_target_years_2001_2020': bool(final['year'].between(YEAR_MIN,YEAR_MAX).all()),
    'feature_matrix_has_no_id_or_country_predictors': not (set(MODEL_FEATURES) & {'UrbanId','Country','country'}),
    'transfer_results_created': results_path.exists() and len(results_df) > 0,
    'all_baseline_models_reported': bool(len(results_df) > 0 and set(results_df['model']) == {'source_mean','ridge','rf'}),
    'test_city_overlap_zero': bool((results_df['city_overlap'] == 0).all()) if len(results_df) else False,
    'composite_validation_and_high_residual_report_written': bool(validation_path.exists() and summary_path.exists() and high_residual_path.exists() and high_residuals_logged),
    'precomputed_jja_target_rule_followed': JJA_TARGET_RULE_APPROVED_BY_PROTOCOL,
}
status = 'GO' if all(checks.values()) else 'NO-GO / REVIEW REQUIRED'
check_lines = [f"- [{'x' if ok else ' '}] {name}" for name,ok in checks.items()]
summary_text = f"""# Phase 2 Core Decision Log

- Run ID: `{RUN_ID}`
- Decision: **{status}**
- Target: `{TARGET_PROD}` / `{TARGET_TIME}` / `{TARGET_COL}` / period `{TARGET_PERIOD}`
- Year range: {YEAR_MIN}–{YEAR_MAX}
- Eligible manifest counts: {eligible_manifest['Country'].value_counts().to_dict()}
- Clean target rows: {len(final):,}
- Clean target unique city-year keys: {final[['UrbanId','year']].drop_duplicates().shape[0]:,}

## Checks
{chr(10).join(check_lines)}

## Interpretation
- “GO” means the machine-checkable core data gates passed. The period-22 precomputed JJA source remains authoritative; the monthly-mean comparison is diagnostic and flagged residuals remain visible in audit outputs.
- The external feature QA and external-feature transfer experiment are covered by notebooks 02 and 03 and must pass their own gates.
- Do not interpret this log as evidence that remote-source values have already been re-downloaded or that all assumptions have been scientifically reviewed.
"""
(DOCS_DIR / 'DECISION_LOG_CORE.md').write_text(summary_text, encoding='utf-8')
print(summary_text)

# Phase 2 Core Decision Log

- Run ID: `20261018_002711`
- Decision: **GO**
- Target: `Mod2` / `Nig` / `Intensity_MEA` / period `22`
- Year range: 2001–2020
- Eligible manifest counts: {'China': 1441, 'Türkiye': 180}
- Clean target rows: 32,420
- Clean target unique city-year keys: 32,420

## Checks
- [x] manifest_unique_UrbanId
- [x] country_year_coverage_table_complete
- [x] final_target_rows_reconcile_to_logged_exclusions
- [x] disputed_list_has_13_ids
- [x] all_disputed_ids_quarantined_and_absent_from_final
- [x] all_expected_seasonal_files_present
- [x] all_expected_monthly_files_present
- [x] primary_target_has_only_period_22
- [x] primary_target_has_only_Mod2_Nig
- [x] clean_target_unique_UrbanId_year
- [x] clean_target_complete_labels
- [x] clean_target_years_2001_2020
- [x] feature_matrix_has_no_id_or_country_predictors
- [x] transfer_results_created
- [x] all_baseline_models_reported
- [x] test_city_overlap_zero
- [x] composite_validation_and_high_residual_report_written
- [x